# Ensemble forecast explorer (IFS analysis as truth)

Visualises an ensemble run from this pipeline, verified against Brightband's
ECMWF IFS analysis on Arraylake rather than ERA5.

Why IFS: AIFS and Aurora 1.5 are both fine-tuned on IFS operational
analyses, so IFS is the analysis they were built to reproduce. ERA5 is the
one Aurora 1 was trained on. Scoring against both is the honest thing to do;
this notebook is the IFS half.

Caveat: the IFS repo is a rolling ~2-week window, so only recent init times
exist. Historical dates are ERA5-only.

Prerequisite — an ensemble forecast on disk:

    uv run scripts/run_benchmark.py --models aifs-ens1.0 --n-members 10 \
        --init-times <RECENT_DATE> --n-steps 8 --variable t2m

## Configuration

In [4]:
from arraylake import Client

client = Client()
client.login()

IFS_REPO   = "emmahf/ecmwf-ifs-initial-conditions-open-subscription"

Successfully refreshed tokens! Token stored at /admin/home/emma_scharfmann/.arraylake/token.json

╭───────────────────────────────────────────────── User Details ──────────────────────────────────────────────────╮
│ Name: Emma Scharfmann                                                                                           │
│ Email: emma.scharfmann@huggingface.co                                                                           │
│ Id: 444b8da7-fe0d-4a8f-a320-305d8e7b2207                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
MODEL      = "aifs-ens1.0"
INIT_TIME  = "2026-09-14T00:00:00"   # must be inside the IFS rolling window
VARIABLE   = "t2m"

SITE_NAME  = "Zurich"
SITE_LAT, SITE_LON = 47.37, 8.55     # lon in 0-360
THRESHOLD  = 288.15                  # Kelvin: 288.15 K = 15 C



In [6]:
import sys
sys.path.insert(0, "..")

import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt

from weather_pipeline.postprocess import load_forecast
from weather_pipeline.scoring import ensemble_dim



plt.rcParams["figure.dpi"] = 110

# Forecast variable id -> IFS repo variable name. The repo uses GRIB-ish
# short names that differ from the pipeline's Earth2Studio-style ids.
IFS_NAMES = {"t2m": "t2m", "u10m": "u10", "v10m": "v10",
             "d2m": "d2m", "msl": "msl", "skt": "skt", "sp": "sp"}

def open_ifs(repo_name=IFS_REPO):
    """Open the IFS analysis. Longitude is stored -180..180 and has to be
    rolled to 0..360 to line up with the forecasts."""
    repo = Client().get_repo(repo_name)
    ds = xr.open_zarr(repo.readonly_session("main").store,
                      zarr_format=3, consolidated=False, chunks=None)
    if float(ds.longitude.min()) < 0:
        ds = ds.assign_coords(longitude=ds.longitude % 360).sortby("longitude")
    return ds

ifs = open_ifs()
times = np.sort(ifs.init_time.values)
print(f"IFS window: {times[0]} .. {times[-1]}  ({len(times)} init times)")

if np.datetime64(pd.Timestamp(INIT_TIME)) not in times:
    print(f"\n!! {INIT_TIME} is NOT in the window. Pick one of the last few:")
    print("   " + "\n   ".join(str(t) for t in times[-6:]))

IFS window: 2026-08-31T00:00:00.000000000 .. 2026-09-15T06:00:00.000000000  (62 init times)


## Load the forecast and ERA5 truth

In [7]:
slug = pd.Timestamp(INIT_TIME).strftime("%Y%m%dT%H%M")
fc = load_forecast(f"../outputs/forecasts/{MODEL}_{slug}.zarr", [VARIABLE])[VARIABLE]

ENS = ensemble_dim(fc)
if ENS is None:
    raise SystemExit("No ensemble dimension — re-run with --n-members > 1.")

lead_h = (fc["lead_time"].values / np.timedelta64(1, "h")).astype(int)
print(f"{fc.sizes[ENS]} members, lead times {list(lead_h)} h")

def fetch_ifs_truth(ifs, variable, valid_times):
    """Analysis at each valid time = that init_time at lead_time 0.

    An analysis, not a forecast: lead_time=0 is the model's best estimate of
    the atmosphere at that moment, which is what verification wants."""
    name = IFS_NAMES.get(variable, variable)
    frames = []
    for vt in pd.to_datetime(list(valid_times)):
        frames.append(
            ifs[name].sel(init_time=np.datetime64(vt),
                          lead_time=np.timedelta64(0, "ns"), drop=True)
        )
    out = xr.concat(frames, dim="valid_time")
    return out.assign_coords(valid_time=list(valid_times))

truth = fetch_ifs_truth(ifs, VARIABLE, fc["valid_time"].values).load()

# align grids: forecasts are descending-lat / 0-360 lon
if float(truth.latitude[0]) < float(truth.latitude[-1]):
    truth = truth.isel(latitude=slice(None, None, -1))
truth = truth.interp(latitude=fc.latitude, longitude=fc.longitude)
print("truth:", dict(truth.sizes))

FileNotFoundError: /fsx/emma_scharfmann/outputs/forecasts/aifs-ens1.0_20260914T0000.zarr does not exist

## Plot

In [ ]:
def at_point(da, lat, lon):
    return da.sel(latitude=lat, longitude=lon % 360, method="nearest")

fc_pt, truth_pt = at_point(fc, SITE_LAT, SITE_LON), at_point(truth, SITE_LAT, SITE_LON)

fig, ax = plt.subplots(figsize=(8, 4.5))
for m in range(fc_pt.sizes[ENS]):
    ax.plot(lead_h, fc_pt.isel({ENS: m}).values, color="steelblue", alpha=0.45, lw=1)
ax.plot(lead_h, fc_pt.mean(dim=ENS).values, "navy", lw=2.5, label="ensemble mean")
ax.plot(lead_h, truth_pt.values, "crimson", ls="--", lw=2.5, label="IFS analysis")
ax.plot([], [], color="steelblue", alpha=0.45, label="members")
ax.set_xlabel("Lead time [h]"); ax.set_ylabel(VARIABLE)
ax.set_title(f"{MODEL} — {SITE_NAME} — init {INIT_TIME}")
ax.grid(alpha=0.3); ax.legend(); plt.show()

In [ ]:
mean = fc_pt.mean(dim=ENS).values
std  = fc_pt.std(dim=ENS, ddof=1).values
p10  = fc_pt.quantile(0.10, dim=ENS).values
p90  = fc_pt.quantile(0.90, dim=ENS).values

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(lead_h, mean - std, mean + std, color="steelblue", alpha=0.25,
                label="mean ± 1 sd")
ax.errorbar(lead_h, mean, yerr=[mean - p10, p90 - mean], fmt="o-", color="navy",
            capsize=4, lw=2, label="mean, 10–90th pct")
ax.plot(lead_h, truth_pt.values, "crimson", ls="--", lw=2.5, label="IFS analysis")

inside = ((truth_pt.values >= p10) & (truth_pt.values <= p90)).mean()
ax.set_title(f"truth inside the 10–90% range {100*inside:.0f}% of the time "
             f"(ideal ≈ 80%)")
ax.set_xlabel("Lead time [h]"); ax.grid(alpha=0.3); ax.legend(); plt.show()

In [ ]:
def spaghetti(lat, lon):
    f, t = at_point(fc, lat, lon), at_point(truth, lat, lon)
    fig, ax = plt.subplots(figsize=(8, 4))
    for m in range(f.sizes[ENS]):
        ax.plot(lead_h, f.isel({ENS: m}).values, color="steelblue", alpha=0.4, lw=1)
    ax.plot(lead_h, f.mean(dim=ENS).values, "navy", lw=2.5)
    ax.plot(lead_h, t.values, "crimson", ls="--", lw=2.5)
    ax.set_title(f"lat {lat:.1f}, lon {lon:.1f}"); ax.grid(alpha=0.3); plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(spaghetti,
             lat=FloatSlider(value=SITE_LAT, min=-88, max=88, step=2),
             lon=FloatSlider(value=SITE_LON, min=0, max=358, step=2))
except ImportError:
    print("ipywidgets not installed — static example instead")
    spaghetti(0.0, 300.0)

In [ ]:
STEP = -1   # which lead time; -1 = last

sel = fc.isel(valid_time=STEP)
mean_map, spread_map = sel.mean(dim=ENS), sel.std(dim=ENS, ddof=1)
err_map = mean_map - truth.isel(valid_time=STEP)
lim = float(abs(err_map).quantile(0.98))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
for ax, (d, title, cmap, kw) in zip(axes, [
    (mean_map,   "ensemble mean",       "RdYlBu_r", {}),
    (spread_map, "spread (sd)",         "viridis",  {}),
    (err_map,    "mean error vs IFS",   "RdBu_r",   dict(vmin=-lim, vmax=lim)),
]):
    im = ax.pcolormesh(d["longitude"], d["latitude"], d.values,
                       cmap=cmap, shading="auto", **kw)
    ax.set_title(f"{title} — +{lead_h[STEP]} h", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout(); plt.show()

In [ ]:
prob = (fc > THRESHOLD).mean(dim=ENS)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
im = axes[0].pcolormesh(prob["longitude"], prob["latitude"],
                        prob.isel(valid_time=STEP).values,
                        cmap="plasma", vmin=0, vmax=1, shading="auto")
axes[0].set_title(f"P({VARIABLE} > {THRESHOLD}) — +{lead_h[STEP]} h", fontsize=10)
fig.colorbar(im, ax=axes[0], shrink=0.85)
axes[1].pcolormesh(prob["longitude"], prob["latitude"],
                   (truth > THRESHOLD).isel(valid_time=STEP).values.astype(float),
                   cmap="plasma", vmin=0, vmax=1, shading="auto")
axes[1].set_title("IFS analysis: exceeded (1) / not (0)", fontsize=10)
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
import io as _io
import imageio.v3 as iio
from IPython.display import Image, display

sp_max = float(fc.std(dim=ENS, ddof=1).quantile(0.99))
v0, v1 = float(fc.quantile(0.02)), float(fc.quantile(0.98))
frames = []

for i in range(fc.sizes["valid_time"]):
    s = fc.isel(valid_time=i)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.1), dpi=100)
    axes[0].pcolormesh(s["longitude"], s["latitude"], s.mean(dim=ENS).values,
                       cmap="RdYlBu_r", vmin=v0, vmax=v1, shading="auto")
    axes[0].set_title(f"ensemble mean — +{lead_h[i]} h", fontsize=10)
    axes[1].pcolormesh(s["longitude"], s["latitude"], s.std(dim=ENS, ddof=1).values,
                       cmap="viridis", vmin=0, vmax=sp_max, shading="auto")
    axes[1].set_title(f"spread — +{lead_h[i]} h", fontsize=10)
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    buf = _io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
    buf.seek(0); frames.append(iio.imread(buf))

gif_path = f"../outputs/{MODEL}_{VARIABLE}_{slug}_ensemble_ifs.gif"
iio.imwrite(gif_path, frames, duration=600, loop=0)
display(Image(filename=gif_path))

In [ ]:
sub = dict(latitude=slice(None, None, 4), longitude=slice(None, None, 4))
f_s, t_s = fc.isel(**sub), truth.isel(**sub)

ranks = (f_s < t_s).sum(dim=ENS).values.ravel()
n_bins = f_s.sizes[ENS] + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(ranks, bins=np.arange(n_bins + 1) - 0.5, color="steelblue", edgecolor="white")
axes[0].axhline(len(ranks) / n_bins, color="crimson", ls="--", label="flat = calibrated")
axes[0].set_xlabel("rank of truth among members"); axes[0].set_title("Rank histogram")
axes[0].legend(fontsize=8)

w = np.cos(np.deg2rad(fc["latitude"]))
rmse   = np.sqrt(((fc.mean(dim=ENS) - truth) ** 2).weighted(w)
                 .mean(dim=("latitude", "longitude"))).values
spread = np.sqrt(fc.var(dim=ENS, ddof=1).weighted(w)
                 .mean(dim=("latitude", "longitude"))).values

axes[1].plot(lead_h, rmse, "o-", label="RMSE of ensemble mean")
axes[1].plot(lead_h, spread, "s-", label="ensemble spread")
axes[1].set_xlabel("Lead time [h]"); axes[1].set_title("Spread vs skill")
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(pd.Series(spread / rmse, index=pd.Index(lead_h, name="lead_h")).round(3))
print("\n~1 = calibrated, <1 = under-dispersed (overconfident)")

In [ ]:
def spaghetti(lat, lon):    f, t = at_point(fc, lat, lon), at_point(truth, lat, lon)    fig, ax = plt.subplots(figsize=(8, 4))    for m in range(f.sizes[ENS]):        ax.plot(lead_h, f.isel({ENS: m}).values, color="steelblue", alpha=0.4, lw=1)    ax.plot(lead_h, f.mean(dim=ENS).values, "navy", lw=2.5)    ax.plot(lead_h, t.values, "crimson", ls="--", lw=2.5)    ax.set_title(f"lat {lat:.1f}, lon {lon:.1f}")    ax.set_xlabel("Lead time [h]"); ax.set_ylabel(VARIABLE); ax.grid(alpha=0.3)    plt.show()try:    from ipywidgets import interact, FloatSlider    interact(spaghetti,             lat=FloatSlider(value=SITE_LAT, min=-88, max=88, step=2),             lon=FloatSlider(value=SITE_LON, min=0, max=358, step=2))except ImportError:    print("ipywidgets not installed — showing a static example instead")    spaghetti(0.0, 300.0)

In [ ]:
STEP = -1           # which lead time to map; -1 = the last onesel = fc.isel(valid_time=STEP)mean_map, spread_map = sel.mean(dim=ENS), sel.std(dim=ENS, ddof=1)err_map = mean_map - truth.isel(valid_time=STEP)fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))for ax, (data, title, cmap, kw) in zip(axes, [    (mean_map,   f"ensemble mean",       "RdYlBu_r", {}),    (spread_map, "ensemble spread (sd)", "viridis",  {}),    (err_map,    "mean error vs truth",  "RdBu_r",     dict(vmin=-float(abs(err_map).quantile(0.98)),          vmax=float(abs(err_map).quantile(0.98)))),]):    im = ax.pcolormesh(data["longitude"], data["latitude"], data.values,                       cmap=cmap, shading="auto", **kw)    ax.set_title(f"{title} — +{lead_h[STEP]} h", fontsize=10)    ax.set_xticks([]); ax.set_yticks([])    fig.colorbar(im, ax=ax, shrink=0.85)plt.tight_layout(); plt.show()

In [ ]:
# A few individual members, shared colour scale: how different are the# plausible outcomes from one another?n_show = min(4, fc.sizes[ENS])vmin, vmax = float(sel.quantile(0.02)), float(sel.quantile(0.98))fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 2.9))for i, ax in enumerate(np.atleast_1d(axes)):    d = sel.isel({ENS: i})    im = ax.pcolormesh(d["longitude"], d["latitude"], d.values,                       cmap="RdYlBu_r", vmin=vmin, vmax=vmax, shading="auto")    ax.set_title(f"member {i}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])fig.colorbar(im, ax=axes, shrink=0.8)plt.show()